In [4]:
import os
import glob
import arcpy
from arcpy.sa import *
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput        = True
arcpy.env.outputCoordinateSystem = None

# Mosaic smaller rasters into one that covers Illinois

In [5]:
# folder with separate rasters
raster_folder = r"D:\Databases\carbon\Soil_Rasters\gNATSGO"

In [6]:
# output folder
output_location = r"D:\Databases\carbon\Soil_Rasters"
output_name = "gNATSGO_soc_0_100cm_g_m2_mosaic.tif"

In [ ]:
# collect all rasters
rasters = glob.glob(os.path.join(raster_folder, "*.tif"))
print(f"Found {len(rasters)} rasters.")

In [ ]:
# get CRS and cell size from first raster
r0   = arcpy.Raster(rasters[0])
sr   = r0.spatialReference
cell = r0.meanCellWidth

In [ ]:
# perform mosaic
arcpy.management.MosaicToNewRaster(
    input_rasters=rasters,
    output_location=output_location,
    raster_dataset_name_with_extension=output_name,
    coordinate_system_for_the_raster=sr,
    pixel_type="32_BIT_FLOAT",
    cellsize=str(cell),
    number_of_bands=1,
    mosaic_method="FIRST",
    mosaic_colormap_mode="FIRST"
);

In [ ]:
# verify output
r_out = arcpy.Raster(os.path.join(output_location, output_name))
print(f"Output cell size: {r_out.meanCellWidth}")
print(f"Output extent:    {r_out.extent}")
print(f"Output CRS:       {r_out.spatialReference.name}")
print("Done.")

# Project raster and update units

In [ ]:
# Define variables
input_raster = os.path.join(output_location,"gNATSGO_soc_0_100cm_g_m2_mosaic.tif")
output_raster = os.path.join(output_location,"gNATSGO_soc_0_100cm_g_m2_mosaic_WGS1984.tif")
out_coordinate_system = arcpy.SpatialReference(4326) # Example: WGS 84 / UTM zone 14N
resampling_method = "BILINEAR" # Use "NEAREST" for categorical data

In [ ]:
# Run the Project Raster tool
arcpy.management.ProjectRaster(
    in_raster=input_raster,
    out_raster=output_raster,
    out_coor_system=out_coordinate_system,
    resampling_type=resampling_method
);

In [ ]:
# Update raster units: (g/m2)/100 = t/ha
raster_t_ha = RasterCalculator([output_raster],["x"], "x/100")

In [ ]:
# Save the output raster permanently to disk
raster_t_ha.save(os.path.join(output_location,r"gNATSGO_soc_0_100cm_t_ha_mosaic_WGS1984.tif"))

# Resample SoilGrids raster

In [9]:
# make gNATSGO raster the reference
reference = os.path.join(output_location,r"gNATSGO_soc_0_100cm_t_ha_mosaic_WGS1984.tif")
ref_r     = arcpy.Raster(reference)
ref_sr    = ref_r.spatialReference
ref_cell  = ref_r.meanCellWidth

print(f"\nReference raster:    {os.path.basename(reference)}")
print(f"Reference cell size: {ref_cell}m")
print(f"Reference CRS:       {ref_r.spatialReference.name}")


Reference raster:    soc_0_100cm_t_ha_mosaic_WGS1984.tif
Reference cell size: 0.00010961388556510059m
Reference CRS:       GCS_WGS_1984


In [11]:
# Set snap raster to reference — forces grid alignment
arcpy.env.snapRaster             = reference
arcpy.env.outputCoordinateSystem = None
resample_method = "BILINEAR"  # continuous

In [12]:
# resample soil grids
arcpy.management.Resample(
    in_raster=os.path.join(output_location,r"SoilGrids\SoilGrids2_0_IL_Clip_ZeroFill.tif"),
    out_raster=os.path.join(output_location,r"SoilGrids\SoilGrids2_0_IL_Clip_ZeroFill_gNATSGO_Resample.tif"),
    cell_size=str(ref_cell),
    resampling_type=resample_method
);

<Result 'D:\\Databases\\carbon\\Soil_Rasters\\SoilGrids2_0_IL_Clip_ZeroFill_gNATSGO_Resample.tif'>

# Fill gNATSO raster with SoilGrids raster

In [13]:
gnatsgo_raster = arcpy.Raster(os.path.join(output_location,"gNATSGO_soc_0_100cm_t_ha_mosaic_WGS1984.tif"))

In [14]:
soil_grids_raster = arcpy.Raster(os.path.join(output_location,"SoilGrids\SoilGrids2_0_IL_Clip_ZeroFill_gNATSGO_Resample.tif"))

In [15]:
less_zero_fill_raster = Con(gnatsgo_raster < 0, soil_grids_raster, gnatsgo_raster)

In [16]:
less_zero_fill_path = os.path.join(output_location,"gNATSGO_soc_0_100cm_t_ha_mosaic_WGS1984_zero_fill.tif")

In [17]:
less_zero_fill_raster.save(less_zero_fill_path)

Done.


In [14]:
less_zero_fill_raster = arcpy.Raster(os.path.join(output_location,"gNATSGO_soc_0_100cm_t_ha_mosaic_WGS1984_zero_fill.tif"))

In [15]:
null_fill_raster = Con(IsNull(less_zero_fill_raster), soil_grids_raster, less_zero_fill_raster)

In [16]:
null_fill_path = os.path.join(output_location,"gNATSGO_soc_0_100cm_t_ha_mosaic_WGS1984_zero_null_fill.tif")

In [17]:
null_fill_raster.save(null_fill_path)

Done.
